# PPLSTM + mLSTM: 10M language tokens + 5,000-step memory probes (Kaggle)

**Same four-block research model as the original Kaggle run:** **mLSTM → PPLSTM → mLSTM → PPLSTM**. The PPLSTM equations and scan remain unchanged.

**Two independent budgets (don't confuse them):**

- **Language model:** 10,000,000 scored next-token *presentations* on the attached `cartman` FineWeb-Edu shard. With batch 8 × context 128, this takes **9,766 optimizer steps** (last step scores 640 tokens). It does **not** mean 10 million unique dataset tokens.
- **Memory probes:** separately initialize and train **three** instances of the same four-block backbone, **5,000 optimizer steps per task**: binary parity, modulo-4 state tracking, and delayed first-symbol recall. Total probe optimizer steps: **15,000**. These probes are *not* evaluations of the text-trained checkpoint.

**What is checked:** scan output/gradient parity, causality, finite gradients, CUDA Graph/eager parity, validation/test PPL and accuracy, throughput, full synthetic-task learning curves and length-generalization accuracy at 64, 128, 256, 512 tokens. The notebook exports CSV/JSON and saves resumable model/probe checkpoints.

**Hardware:** Kaggle accelerator **GPU T4 ×2**, using one T4. This uses **PyTorch CUDA Graph capture of forward + backward**, with gradient clipping and AdamW outside the graph; no custom fused CUDA kernel, no official optimized sLSTM kernel and no sLSTM baseline.

**Data:** attach your Kaggle dataset `rick3yrick3y/cartman` under **Add Input**. The notebook is configured to **stop rather than silently fall back to Tiny Shakespeare**. The FineWeb-Edu tokenizer and preprocessing remain the same as in your previous Kaggle notebook (12M prepared training tokens, sampled/reused for training).

**Important:** Run all cells in order. 5,000 steps × three separate probes takes longer than the earlier 350-step run. Checkpoints are saved in `/kaggle/working/pplstm_10m_memory_5000step/`.

To force a **fresh** run instead of resuming, remove the saved checkpoints in this notebook’s output folder before rerunning.


## 1 · Configuration and dependencies

In [ ]:
# SETTINGS — change these before Run All.
TARGET_TOKENS = 10_000_000     # scored token presentations, not necessarily distinct tokens
WIDTH, DEPTH, HEADS = 72, 4, 4
CONTEXT, BATCH = 128, 8
SEED = 20261115
TRAIN_GRAPH_REQUIRED = True      # True: do not silently fall back to eager LM training
TRAIN_LM = True
RUN_PROBES = True
PROBE_STEPS = 5_000               # each of parity, state tracking, delayed recall
PROBE_TRAIN_CONTEXT = 128
PROBE_BATCH = 16
PROBE_EVAL_CONTEXTS = (64, 128, 256, 512)
PROBE_REPORT_EVERY = 500         # 10 saved/evaluated milestones per task
PROBE_FINAL_EVAL_BATCHES = 32   # 512 held-out examples per context, up from 192 in the prior run
PROBE_SAVE_EVERY = 500         # save each task's weights and optimizer for recovery
PROBE_GRAPH_REQUIRED = False     # if probe-specific capture fails, mark eager explicitly
EVAL_EVERY_TOKENS = 1_000_000
SAVE_EVERY_TOKENS = 5_000_000
SMALL_VAL_TOKENS = 8_192
FINAL_VAL_TOKENS = 65_536
STARTUP_SMOKE_ONLY = False      # True: just correctness + mini benchmark (no long training)

# DATA: 'auto', 'prepared', 'fineweb_parquet', or 'tiny_shakespeare'
DATA_MODE = 'fineweb_parquet'  # require the attached cartman parquet: no silent Tiny Shakespeare fallback
PREPARED_DIR = ''                # directory containing train.bin/valid.bin/test.bin/spm.model
PARQUET_GLOB = '/kaggle/input/**/000_00000.parquet'  # attach a single suitable FineWeb-Edu shard
MAX_PREP_TRAIN_TOKENS = 12_000_000    # same prepared-data cap as original Kaggle 50M run
TEXT_FILE = ''                   # optional attached plain UTF-8 file; used in auto mode
RESUME_CHECKPOINT = ''           # optional path to latest.pt from an attached Kaggle output

from pathlib import Path
import os, sys, subprocess, importlib.util
OUT = Path('/kaggle/working/pplstm_10m_memory_5000step' if Path('/kaggle/working').exists() else '/mnt/data/pplstm_10m_memory_5000step')
OUT.mkdir(parents=True, exist_ok=True)
need=[]
if importlib.util.find_spec('xlstm') is None: need.append('xlstm==2.0.6')
if importlib.util.find_spec('sentencepiece') is None: need.append('sentencepiece')
if importlib.util.find_spec('pyarrow') is None: need.append('pyarrow')
if need:
    print('Installing:', need)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])

import copy, csv, glob, hashlib, json, math, random, statistics, time, traceback, urllib.request, zipfile
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import pandas as pd

assert torch.cuda.is_available(), 'Turn on Kaggle Settings → Accelerator → GPU (T4 x2), then rerun.'
DEVICE=torch.device('cuda:0')
torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print('GPU:',torch.cuda.get_device_name(0),'capability:',torch.cuda.get_device_capability(0))
print('torch:',torch.__version__,'output:',OUT)

def safe_json_write(p, obj):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.pending')
    tmp.write_text(json.dumps(obj,indent=2,default=str))
    os.replace(tmp,p)

def atomic_torch_save(obj,p):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.pending')
    torch.save(obj,tmp)
    os.replace(tmp,p)

def save_csv(p,rows):
    if rows: pd.DataFrame(rows).to_csv(p,index=False)

def export_metric_bundle():
    # Refresh after every milestone; the results survive even if later training stops.
    archive=OUT/'PPLSTM_Kaggle_10M_5000Step_Memory_Results.zip'
    temporary=OUT/(archive.name+'.pending')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for file in list(OUT.glob('*.csv'))+list(OUT.glob('*.json')):
            z.write(file,arcname=file.name)
    os.replace(temporary,archive)
    return archive

def params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 2 · Original PPLSTM cell, including serial reference

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Adapted from the original PPLSTM project by Rickey (https://github.com/rickey1990/PPLSTM/blob/main/pplstm.py)
# The transition equations and scan implementation are kept as published.
"""PPLSTM: associative 3-channel permutation-mixed recurrent cell."""
import itertools
import math
import torch
from torch import Tensor, nn
from torch.nn import functional as F


def shift_right(x: Tensor) -> Tensor:
    return torch.cat((torch.zeros_like(x[:, :1]), x[:, :-1]), dim=1)


def scalar_affine_scan(a: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    A, B = a, b
    step = 1
    while step < a.shape[1]:
        A, B = (
            torch.cat((A[:, :step], A[:, step:] * A[:, :-step]), dim=1),
            torch.cat((B[:, :step], B[:, step:] + A[:, step:] * B[:, :-step]), dim=1),
        )
        step *= 2
    return A, B


def matrix_affine_scan(A: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    step = 1
    while step < A.shape[1]:
        next_A = torch.cat((A[:, :step], A[:, step:] @ A[:, :-step]), dim=1)
        next_b = torch.cat((
            b[:, :step],
            b[:, step:] + (A[:, step:] @ b[:, :-step].unsqueeze(-1)).squeeze(-1),
        ), dim=1)
        A, b = next_A, next_b
        step *= 2
    return A, b


class PPLSTM(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        if width < 3 or width % 3 != 0:
            raise ValueError('width must be a positive multiple of 3')
        self.width = width
        self.groups = width // 3
        self.xproj = nn.Linear(width, 4 * width)
        self.mix_logit = nn.Parameter(torch.full((4, width), math.log(3.0)))
        self.sign_proj = nn.Linear(width, width)
        self.initial_cell = nn.Parameter(torch.linspace(-1.5, 1.5, 3).repeat(self.groups))
        self.input_bias = nn.Parameter(torch.full((width,), -4.0))
        self.forget_bias = nn.Parameter(torch.full((width,), 4.0))
        nn.init.zeros_(self.sign_proj.weight)
        nn.init.constant_(self.sign_proj.bias, 2.8)

        bank = []
        for order in itertools.permutations(range(3)):
            matrix = torch.zeros(3, 3)
            for row, column in enumerate(order):
                matrix[row, column] = 1.0
            bank.append(matrix)
        self.register_buffer('permutation_bank', torch.stack(bank))

        self.selector = nn.Linear(width, self.groups * 6)
        nn.init.zeros_(self.selector.weight)
        nn.init.constant_(self.selector.bias, -1.0)
        with torch.no_grad():
            for group in range(self.groups):
                self.selector.bias[group * 6] = 3.0

    def _token_maps(self, x: Tensor):
        if x.ndim != 3 or x.shape[-1] != self.width:
            raise ValueError('Expected x with shape [batch, time, width]')
        batch, length, width = x.shape
        group_count = self.groups
        previous = shift_right(x)
        coeff = self.mix_logit.sigmoid().view(1, 1, 4, width)
        gates = (self.xproj(x).reshape(batch, length, 4, width) * coeff
                 + self.xproj(previous).reshape(batch, length, 4, width) * (1 - coeff))
        input_logits, forget_logits, candidate_logits, output_logits = gates.unbind(2)
        mixed = x * self.mix_logit[0].sigmoid() + previous * (1 - self.mix_logit[0].sigmoid())
        signs = self.sign_proj(mixed).tanh().reshape(batch, length, group_count, 3)
        z = candidate_logits.tanh().reshape(batch, length, group_count, 3)
        log_i = (input_logits + self.input_bias).reshape(batch, length, group_count, 3).mean(-1)
        log_f = F.logsigmoid((forget_logits + self.forget_bias)
                           .reshape(batch, length, group_count, 3).mean(-1))
        f_prefix = log_f.cumsum(dim=1)
        m = f_prefix + torch.cummax(torch.cat((
            torch.zeros_like(f_prefix[:, :1]), log_i - f_prefix,
        ), dim=1), dim=1).values[:, 1:]
        f = (log_f + shift_right(m) - m).exp()
        i = (log_i - m).exp()
        weights = self.selector(mixed).reshape(batch, length, group_count, 6).softmax(-1)
        P = torch.einsum('btgp,pij->btgij', weights, self.permutation_bank)
        A = f[..., None, None] * P * signs[..., None, :]
        b = i[..., None] * z
        return A, b, f, i, output_logits

    def forward(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        prefix_A, prefix_b = matrix_affine_scan(A, b)
        c0 = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        c = (prefix_A @ c0[:, None, :, :, None]).squeeze(-1) + prefix_b
        norm_A, norm_b = scalar_affine_scan(f, i)
        n = norm_A + norm_b
        ratio = (c / n[..., None].clamp_min(1e-12)).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

    def serial_reference(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        c = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        n = x.new_ones(batch, self.groups)
        states = []
        for t in range(length):
            c = (A[:, t] @ c.unsqueeze(-1)).squeeze(-1) + b[:, t]
            n = f[:, t] * n + i[:, t]
            states.append(c / n[..., None].clamp_min(1e-12))
        ratio = torch.stack(states, dim=1).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

## 3 · Four-block mLSTM + PPLSTM language model (original 50M topology)

In [ ]:
from xlstm import (xLSTMBlockStack, xLSTMBlockStackConfig, mLSTMBlockConfig,
                   mLSTMLayerConfig, sLSTMBlockConfig, sLSTMLayerConfig, FeedForwardConfig)

class PPLSTMReplacementLayer(nn.Module):
    """Adapt original sLSTM layer slot; retain convolution, group norm and dropout."""
    def __init__(self, original_layer):
        super().__init__()
        self.config = original_layer.config
        self.conv1d = original_layer.conv1d if self.config.conv1d_kernel_size > 0 else None
        self.conv_act_fn = original_layer.conv_act_fn if self.conv1d is not None else None
        self.group_norm = original_layer.group_norm
        self.dropout = original_layer.dropout
        self.pplstm = PPLSTM(self.config.embedding_dim)

    def forward(self, x, **kwargs):
        B,S,W=x.shape
        conv_x=self.conv_act_fn(self.conv1d(x)) if self.conv1d is not None else x
        y=self.dropout(self.pplstm(conv_x))
        H=self.config.num_heads
        y=y.reshape(B,S,H,W//H).transpose(1,2)
        y=self.group_norm(y).transpose(1,2).reshape(B,S,W)
        return y


def build_backbone(width=WIDTH, context=CONTEXT):
    assert width%3==0 and width%HEADS==0
    cfg=xLSTMBlockStackConfig(
        mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig(
            conv1d_kernel_size=4, qkv_proj_blocksize=4, num_heads=HEADS)),
        slstm_block=sLSTMBlockConfig(slstm=sLSTMLayerConfig(
            backend='vanilla', num_heads=HEADS, conv1d_kernel_size=4,
            bias_init='powerlaw_blockdependent'),
            feedforward=FeedForwardConfig(proj_factor=1.3, act_fn='gelu')),
        context_length=context, num_blocks=DEPTH,
        embedding_dim=width, slstm_at=[1,3])
    stack=xLSTMBlockStack(cfg)
    assert list(stack.config.block_map)==[0,1,0,1]
    for index in (1,3):
        original=stack.blocks[index].xlstm
        if not hasattr(original,'slstm_cell'):
            raise RuntimeError('xLSTM API changed: expected slstm_cell in layer '+str(index))
        stack.blocks[index].xlstm=PPLSTMReplacementLayer(original)
    return stack

class PPLSTMLanguageModel(nn.Module):
    def __init__(self, vocab, width=WIDTH, context=CONTEXT):
        super().__init__()
        self.vocab=vocab
        self.emb=nn.Embedding(vocab,width)
        nn.init.normal_(self.emb.weight,std=0.02)
        self.backbone=build_backbone(width,context)
        self.norm=nn.LayerNorm(width)
        self.out_bias=nn.Parameter(torch.zeros(vocab))
    def forward(self, tokens):
        h=self.norm(self.backbone(self.emb(tokens)))
        return F.linear(h,self.emb.weight,self.out_bias)  # tied LM output weights

class PPLSTMProbeModel(nn.Module):
    """Same four-block backbone, dedicated embedding & final-token classifier."""
    def __init__(self,num_classes,width=WIDTH,max_context=max(PROBE_EVAL_CONTEXTS),vocab=16):
        super().__init__()
        self.emb=nn.Embedding(vocab,width)
        self.backbone=build_backbone(width,max_context)
        self.norm=nn.LayerNorm(width)
        self.head=nn.Linear(width,num_classes)
    def forward(self,tokens):
        return self.head(self.norm(self.backbone(self.emb(tokens))[:,-1]))

model_demo=PPLSTMLanguageModel(256).to(DEVICE)
print('4-block model structure:',[type(model_demo.backbone.blocks[i].xlstm).__name__ for i in range(4)])
print('Parameter count at vocab 256:',params(model_demo))
del model_demo
torch.cuda.empty_cache()

## 4 · Mathematical correctness, causality and gradient stability checks

In [ ]:
DIAGNOSTICS=[]
def record_check(name, passed, **data):
    row={'test':name,'passed':bool(passed),**data}
    DIAGNOSTICS.append(row)
    print(('PASS' if passed else 'FAIL'),name, {k:v for k,v in data.items() if k not in ('traceback',)})
    save_csv(OUT/'diagnostics.csv',DIAGNOSTICS)
    return row

def check_pplstm_scan(width=12,length=17,precision=torch.float64):
    torch.manual_seed(57)
    cell=PPLSTM(width).to(device=DEVICE,dtype=precision)
    x=torch.randn(2,length,width,device=DEVICE,dtype=precision,requires_grad=True)
    parallel=cell(x); serial=cell.serial_reference(x)
    err=(parallel-serial).abs().max().item()
    pa=torch.autograd.grad(parallel.square().sum(),(x,*cell.parameters()),retain_graph=True)
    sa=torch.autograd.grad(serial.square().sum(),(x,*cell.parameters()))
    grad_err=max((a-b).abs().max().item() for a,b in zip(pa,sa))
    tol=1e-9 if precision==torch.float64 else 1e-4
    record_check('parallel_vs_serial_scan_'+str(precision).split('.')[-1],
                 err<tol and grad_err<tol*10,output_max_error=err,grad_max_error=grad_err,length=length)
    assert DIAGNOSTICS[-1]['passed'],'Parallel recurrence numerical check failed'

check_pplstm_scan(12,17,torch.float64)
check_pplstm_scan(48,128,torch.float32)

# A causal model must not change outputs at earlier positions if later token IDs change.
torch.manual_seed(91)
causal_model=PPLSTMLanguageModel(256).to(DEVICE).eval()
a=torch.randint(0,256,(2,48),device=DEVICE)
b=a.clone(); b[:,24:]=torch.randint(0,256,(2,24),device=DEVICE)
with torch.no_grad():
    ya=causal_model(a)[:,:24]; yb=causal_model(b)[:,:24]
causal_error=(ya-yb).abs().max().item()
record_check('full_stack_future_token_causality',causal_error<1e-5,max_error=causal_error)
assert DIAGNOSTICS[-1]['passed'],'Full model failed causality test'
del causal_model,a,b,ya,yb

torch.manual_seed(92)
finite_model=PPLSTMLanguageModel(256).to(DEVICE)
x=torch.randint(0,256,(2,128),device=DEVICE)
target=torch.randint(0,256,(2,128),device=DEVICE)
loss=F.cross_entropy(finite_model(x).reshape(-1,256),target.reshape(-1))
loss.backward()
all_finite=bool(torch.isfinite(loss).item()) and all(p.grad is not None and bool(torch.isfinite(p.grad).all().item()) for p in finite_model.parameters())
record_check('full_stack_finite_loss_and_gradients',all_finite,loss=float(loss.item()),parameters=params(finite_model))
assert all_finite,'Full model produced missing or nonfinite gradients'
del finite_model,x,target,loss
torch.cuda.empty_cache()

## 5 · CUDA Graph training step and eager equivalence gate

In [ ]:
# CUDA Graph records forward+backward on fixed [batch, context] tensors.
# AdamW step and clipping remain outside capture. This is CUDA Graph replay,
# NOT a new fused CUDA kernel; the PPLSTM scan is the same as above.

def task_loss(logits,labels):
    if logits.ndim==3: return F.cross_entropy(logits.reshape(-1,logits.shape[-1]),labels.reshape(-1),ignore_index=-100)
    return F.cross_entropy(logits,labels)

class CapturedForwardBackward:
    def __init__(self,model,example_x,example_y):
        assert example_x.device.type=='cuda'
        self.model=model
        self.x=torch.zeros_like(example_x)
        self.y=torch.zeros_like(example_y)
        self.graph=torch.cuda.CUDAGraph()
        self.model.train()
        side=torch.cuda.Stream(device=example_x.device)
        side.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(side):
            for _ in range(3):
                self.model.zero_grad(set_to_none=True)
                task_loss(self.model(self.x),self.y).backward()
        torch.cuda.current_stream().wait_stream(side)
        self.model.zero_grad(set_to_none=True)
        with torch.cuda.graph(self.graph,stream=side):
            self.loss=task_loss(self.model(self.x),self.y)
            self.loss.backward()
        # Keep graph-owned gradient buffers alive; don't set grads to None.

    def replay(self,x,y):
        if x.shape!=self.x.shape or y.shape!=self.y.shape:
            raise ValueError('CUDA Graph shape mismatch')
        self.x.copy_(x); self.y.copy_(y)
        self.graph.replay()
        return self.loss


def update_graph_step(captured,optimizer,x,y,lr=None):
    if lr is not None:
        for group in optimizer.param_groups: group['lr']=float(lr)
    loss=captured.replay(x,y)
    torch.nn.utils.clip_grad_norm_(captured.model.parameters(),1.0)
    optimizer.step()
    return loss


def update_eager_step(model,optimizer,x,y,lr=None):
    if lr is not None:
        for group in optimizer.param_groups: group['lr']=float(lr)
    optimizer.zero_grad(set_to_none=True)
    loss=task_loss(model(x),y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    optimizer.step()
    return loss


def graph_eager_gate(vocab=256):
    # Two separately seeded replicas start with exactly the same weights.
    torch.manual_seed(111)
    eager=PPLSTMLanguageModel(vocab).to(DEVICE)
    torch.manual_seed(111)
    graphed=PPLSTMLanguageModel(vocab).to(DEVICE)
    assert all(torch.equal(a,b) for a,b in zip(eager.parameters(),graphed.parameters()))
    x=torch.randint(0,vocab,(BATCH,CONTEXT),device=DEVICE)
    y=torch.randint(0,vocab,(BATCH,CONTEXT),device=DEVICE)
    oe=torch.optim.AdamW(eager.parameters(),lr=0.0008,betas=(0.9,0.95),weight_decay=0.01)
    og=torch.optim.AdamW(graphed.parameters(),lr=0.0008,betas=(0.9,0.95),weight_decay=0.01)
    captured=CapturedForwardBackward(graphed,x,y)
    results=[]
    for step in range(2):
        # Unclipped gradient comparison is done before either optimizer update.
        oe.zero_grad(set_to_none=True)
        le=task_loss(eager(x),y);le.backward()
        lg=captured.replay(x,y)
        torch.cuda.synchronize()
        loss_delta=abs(float(le.item())-float(lg.item()))
        gradient_delta=max(float((a.grad-b.grad).abs().max().item()) for a,b in zip(eager.parameters(),graphed.parameters()))
        grad_ok=all(torch.allclose(a.grad,b.grad,atol=5e-5,rtol=5e-4) for a,b in zip(eager.parameters(),graphed.parameters()))
        torch.nn.utils.clip_grad_norm_(eager.parameters(),1.0)
        torch.nn.utils.clip_grad_norm_(graphed.parameters(),1.0)
        oe.step();og.step()
        param_delta=max(float((a-b).abs().max().item()) for a,b in zip(eager.parameters(),graphed.parameters()))
        weights_ok=all(torch.allclose(a,b,atol=5e-5,rtol=5e-4) for a,b in zip(eager.parameters(),graphed.parameters()))
        results.append({'step':step+1,'loss_error':loss_delta,'gradient_max_error':gradient_delta,'post_step_weight_max_error':param_delta,'passed':bool(grad_ok and weights_ok and loss_delta<5e-5)})
        if not results[-1]['passed']: raise AssertionError('CUDA Graph differs from eager at step '+str(step+1))
    # Crucial: graph replay must not accumulate gradient from previous replay.
    captured.replay(x,y)
    first=[p.grad.detach().clone() for p in graphed.parameters()]
    captured.replay(x,y)
    torch.cuda.synchronize()
    overwrite=all(torch.allclose(a,p.grad,atol=5e-5,rtol=5e-4) for a,p in zip(first,graphed.parameters()))
    if not overwrite: raise AssertionError('Captured gradients accumulated on replay')
    record_check('lm_graph_vs_eager_two_steps',True,loss_max_error=max(r['loss_error'] for r in results),
                 gradient_max_error=max(r['gradient_max_error'] for r in results),
                 weight_max_error=max(r['post_step_weight_max_error'] for r in results),gradient_buffers_overwritten=overwrite)
    save_csv(OUT/'graph_eager_parity.csv',results)
    del eager,graphed,oe,og,captured
    torch.cuda.empty_cache()
    return results

try:
    GRAPH_PARITY_RESULTS=graph_eager_gate()
    LM_GRAPH_OK=True
except Exception as ex:
    LM_GRAPH_OK=False
    record_check('lm_graph_vs_eager_two_steps',False,error=f'{type(ex).__name__}: {ex}')
    print('CUDA GRAPH NOT VALIDATED:',str(ex))
    if TRAIN_GRAPH_REQUIRED:
        raise RuntimeError('Graph correctness gate failed. No 10M training was started; review error or set TRAIN_GRAPH_REQUIRED=False.') from ex

## 6 · Learned memory probes: parity, finite-state tracking and delayed recall

**These are separate supervised tasks** using new instances of the **same four-block mLSTM + PPLSTM architecture**. Each trains for `PROBE_STEPS` with independent random examples and evaluates on fresh held-out sequences. It is *not* a pretrained language model's zero-shot evaluation.

| Probe | Input sequence | Output | Chance accuracy |
|---|---|---|---:|
| Binary parity | Random 0/1 bits + query token | XOR of all bits | 50% |
| Finite-state tracking | Increment / decrement / hold / reset | Final state modulo 4 | 25% |
| Delayed recall | Six symbol classes, arbitrary distractors, final query | Identity of **first** symbol | 16.7% |

Train at 128 tokens and evaluate at 64, 128, 256 and 512: this distinguishes learned training-length performance from longer-context generalization. Low parity scores do **not** mean the model has a numerical bug—parity can be difficult to learn.

In [ ]:
# Restore saved learning curves and completed-task metrics when a Kaggle run is resumed.
def _restore_probe_csv(name):
    path=OUT/name
    return pd.read_csv(path).to_dict('records') if path.exists() and path.stat().st_size else []
PROBE_ROWS=_restore_probe_csv('memory_probes.csv')
PROBE_CURVES=_restore_probe_csv('memory_probe_learning_curves.csv')
def synthetic_batch(task,batch,length,seed):
    rng=np.random.default_rng(int(seed))
    if task=='parity':
        bits=rng.integers(0,2,size=(batch,length-1),dtype=np.int64)
        x=np.concatenate([bits,np.full((batch,1),3,dtype=np.int64)],axis=1)
        labels=(bits.sum(axis=1)%2).astype(np.int64)
    elif task=='state_mod4':
        # Stratify candidate examples to give balanced labels (25% chance baseline).
        # Reset is retained in the task; without balancing, state 0 would be overrepresented.
        candidate_count=max(batch*8,64)
        operations=rng.choice(np.array([4,5,6,7],dtype=np.int64),
                              size=(candidate_count,length-1),p=[.3,.3,.3,.1])
        indices=np.arange(length-1)[None,:]
        last_reset=np.where(operations==7,indices,-1).max(axis=1)
        active=indices>last_reset[:,None]
        candidate_labels=((((operations==4)&active).sum(axis=1)-
                          ((operations==5)&active).sum(axis=1))%4).astype(np.int64)
        selected=[]
        for klass in range(4):
            available=np.flatnonzero(candidate_labels==klass)
            need=batch//4+(1 if klass<batch%4 else 0)
            selected.extend(rng.choice(available,size=need,replace=len(available)<need).tolist())
        rng.shuffle(selected)
        labels=candidate_labels[selected]
        chosen=operations[selected]
        x=np.concatenate([chosen,np.full((batch,1),8,dtype=np.int64)],axis=1)
    elif task=='delayed_recall':
        symbols=rng.integers(9,15,size=(batch,length-1),dtype=np.int64)
        labels=(symbols[:,0]-9).astype(np.int64)
        x=np.concatenate([symbols,np.full((batch,1),15,dtype=np.int64)],axis=1)
    else:raise ValueError(task)
    return torch.from_numpy(x).to(DEVICE),torch.from_numpy(labels.astype(np.int64)).to(DEVICE)

@torch.no_grad()
def eval_probe(model,task,length,batches=12):
    model.eval()
    correct=0;tot=0;losses=0.
    for ix in range(batches):
        x,y=synthetic_batch(task,PROBE_BATCH,length,seed=992211+length*43+ix)
        logits=model(x)
        correct+=int((logits.argmax(-1)==y).sum().item())
        tot+=int(y.numel())
        losses+=float(F.cross_entropy(logits,y,reduction='sum').item())
    model.train()
    return {'accuracy_pct':100*correct/tot,'cross_entropy':losses/tot,'val_samples':tot}


def train_probe(task,classes,seed):
    # Each synthetic task trains its OWN independently initialized model.
    torch.manual_seed(seed)
    pmodel=PPLSTMProbeModel(classes).to(DEVICE)
    optimizer=torch.optim.AdamW(pmodel.parameters(),lr=1e-3,weight_decay=0.01)
    probe_ckpt=OUT/f'probe_{task}_latest.pt'
    start_step=0
    elapsed_prior=0.0
    if probe_ckpt.exists():
        saved=torch.load(probe_ckpt,map_location=DEVICE,weights_only=False)
        expected={'task':task,'classes':classes,'seed':seed,'context':PROBE_TRAIN_CONTEXT,'batch':PROBE_BATCH}
        for key,value in expected.items():
            if saved.get(key)!=value:
                raise ValueError(f'Probe checkpoint {probe_ckpt} has mismatching {key}; remove it for a fresh run')
        pmodel.load_state_dict(saved['model']);optimizer.load_state_dict(saved['optimizer'])
        start_step=int(saved['steps']); elapsed_prior=float(saved.get('training_seconds',0))
        print('RESUMING PROBE',task,'from step',start_step,'of',PROBE_STEPS,flush=True)
    train_x,train_y=synthetic_batch(task,PROBE_BATCH,PROBE_TRAIN_CONTEXT,seed+1)
    proc_engine='graph'
    try:
        graph=CapturedForwardBackward(pmodel,train_x,train_y)
    except Exception as exc:
        if PROBE_GRAPH_REQUIRED:raise
        print('Probe',task,'graph unavailable; explicitly using eager:',str(exc))
        proc_engine='eager';graph=None
        pmodel.zero_grad(set_to_none=True)
    t0=time.perf_counter()
    for step_i in range(start_step,PROBE_STEPS):
        x,y=synthetic_batch(task,PROBE_BATCH,PROBE_TRAIN_CONTEXT,seed+10000+step_i)
        if graph:loss=update_graph_step(graph,optimizer,x,y)
        else:loss=update_eager_step(pmodel,optimizer,x,y)
        current_step=step_i+1
        if current_step%PROBE_REPORT_EVERY==0 or current_step==PROBE_STEPS:
            score=eval_probe(pmodel,task,PROBE_TRAIN_CONTEXT,batches=8)
            row={'task':task,'training_step':current_step,'val_context':PROBE_TRAIN_CONTEXT,
                 'accuracy_pct':score['accuracy_pct'],'cross_entropy':score['cross_entropy'],
                 'engine':proc_engine}
            PROBE_CURVES[:]=[r for r in PROBE_CURVES if not
                             (r['task']==task and int(r['training_step'])==current_step)]
            PROBE_CURVES.append(row)
            save_csv(OUT/'memory_probe_learning_curves.csv',PROBE_CURVES)
            print(task,'step',f'{current_step:,}/{PROBE_STEPS:,}',
                  'val_accuracy',round(score['accuracy_pct'],2),
                  'train_loss',round(float(loss.item()),4),'engine',proc_engine,flush=True)
            export_metric_bundle()
        if current_step%PROBE_SAVE_EVERY==0 or current_step==PROBE_STEPS:
            # Atomic save; use Kaggle Save Version or download output to preserve across sessions.
            torch.cuda.synchronize()
            atomic_torch_save({'model':pmodel.state_dict(),'optimizer':optimizer.state_dict(),
              'steps':current_step,'training_seconds':elapsed_prior+time.perf_counter()-t0,
              'task':task,'classes':classes,'seed':seed,
              'context':PROBE_TRAIN_CONTEXT,'batch':PROBE_BATCH,
              'engine':proc_engine},probe_ckpt)
            print('SAVED probe checkpoint',probe_ckpt,'step',current_step,flush=True)
    torch.cuda.synchronize()
    elapsed=elapsed_prior+time.perf_counter()-t0
    PROBE_ROWS[:]=[r for r in PROBE_ROWS if r['task']!=task]  # replace task's final evaluations if rerun
    for length in PROBE_EVAL_CONTEXTS:
        try:
            stats=eval_probe(pmodel,task,length,batches=PROBE_FINAL_EVAL_BATCHES)
            row={'task':task,'train_context':PROBE_TRAIN_CONTEXT,'eval_context':length,
                 'steps':PROBE_STEPS,'engine':proc_engine,'parameters':params(pmodel),
                 'accuracy_pct':stats['accuracy_pct'],'val_cross_entropy':stats['cross_entropy'],
                 'val_examples':stats['val_samples'],'chance_pct':100/classes,
                 'train_eval_elapsed_s':elapsed,'status':'MEASURED'}
        except (RuntimeError,torch.cuda.OutOfMemoryError) as ex:
            row={'task':task,'train_context':PROBE_TRAIN_CONTEXT,'eval_context':length,
                 'steps':PROBE_STEPS,'engine':proc_engine,'status':'FAIL',
                 'error':f'{type(ex).__name__}: {ex}'}
            torch.cuda.empty_cache()
        PROBE_ROWS.append(row)
        print('EVAL',row,flush=True)
        save_csv(OUT/'memory_probes.csv',PROBE_ROWS)
        save_csv(OUT/'memory_probe_learning_curves.csv',PROBE_CURVES)
        export_metric_bundle()
    del pmodel,optimizer,graph
    torch.cuda.empty_cache()

if RUN_PROBES and not STARTUP_SMOKE_ONLY:
    for k,cls,seed in [('parity',2,5001),('state_mod4',4,5002),('delayed_recall',6,5003)]:
        train_probe(k,cls,seed)
elif STARTUP_SMOKE_ONLY:
    for name in ('parity','state_mod4','delayed_recall'):
        x,y=synthetic_batch(name,2,32,2026)
        print('PROBE SMOKE',name,x.shape,y.tolist())

## 7 · Dataset preparation and provenance

In [ ]:
"""Frozen FineWeb-Edu sample-10BT data pipeline.
Only the dataset/test *types* were taken from the provided example notebook.
No model code or experiment results from that notebook are imported.
"""
import glob,hashlib,json,os,time
from pathlib import Path
import numpy as np


def sha_text(x):
    return hashlib.sha256(x.encode('utf8','ignore')).hexdigest()


def split_doc(text):
    """Exact duplicates land in the same split; near-duplicates remain a caveat."""
    r=int(sha_text(text)[:12],16)%1000
    return 'valid' if r<10 else 'test' if r<20 else 'train'


def documents(paths):
    import pyarrow.parquet as pq
    for filename in paths:
        pqfile=pq.ParquetFile(str(filename))
        if 'text' not in pqfile.schema.names:
            raise ValueError(f'Not a FineWeb text Parquet shard: {filename}')
        for batch in pqfile.iter_batches(columns=['text'],batch_size=512):
            for raw in batch.column(0).to_pylist():
                if isinstance(raw,str) and len(raw.strip())>=20:
                    yield raw.replace('\x00',' ').strip()


def find_paths(pattern):
    found=[Path(p) for p in sorted(glob.glob(pattern,recursive=True)) if Path(p).is_file()]
    if not found:
        raise FileNotFoundError('FineWeb-Edu parquet not found. Attach sample/10BT/000_00000.parquet in Kaggle Inputs, then edit PARQUET_GLOB. Current: '+pattern)
    if len(found)>1:
        filtered=[p for p in found if 'fineweb' in str(p).lower()]
        if filtered:found=filtered
        else:raise RuntimeError('Multiple generic parquet shards matched. Narrow PARQUET_GLOB to FineWeb-Edu sample-10BT only.')
    return found


def manifest_for_input(paths):
    rows=[]
    for p in paths:
        with p.open('rb') as stream:head=stream.read(1024*1024)
        rows.append({'path':str(p),'file_size':p.stat().st_size,
             'first_megabyte_sha256':hashlib.sha256(head).hexdigest()})
    return rows


def prepare(root,glob_pattern,existing_data_root='',vocab=4096,max_train=30_000_000,
            valid_count=250_000,test_count=250_000,sample_chars=3_000_000,
            stop_epoch=None):
    import sentencepiece as spm
    out=Path(root)/'data';out.mkdir(parents=True,exist_ok=True)
    if existing_data_root:
        base=Path(existing_data_root)/'data'
        if not (base/'spm.model').exists() or not (base/'train.bin').exists():
            raise ValueError('EXISTING_DATA_ROOT must contain data/spm.model and train/valid/test.bin')
        manifest=json.loads((base/'manifest.json').read_text()) if (base/'manifest.json').exists() else {}
        return {'root':str(base),'tokenizer':str(base/'spm.model'),'sizes':{
            part:(base/f'{part}.bin').stat().st_size//2 for part in ('train','valid','test')},
            'reused':True,'original_manifest':manifest,'source':'FineWeb-Edu, externally prepared corpus (verify uploaded provenance)'}

    old=out/'manifest.json'
    if old.exists() and (out/'spm.model').exists():
        manifest=json.loads(old.read_text())
        if manifest.get('complete') and all((out/f'{k}.bin').exists() for k in ('train','valid','test')):
            return manifest
    paths=find_paths(glob_pattern)
    source_manifest=manifest_for_input(paths)
    sample=out/'spm_training_sample.txt'
    if not (out/'spm.model').exists():
        chars=0;count=0
        with sample.open('w',encoding='utf8') as f:
            for t in documents(paths):
                if split_doc(t)!='train':continue
                t=' '.join(t[:15000].split());f.write(t+'\n');chars+=len(t);count+=1
                if chars>=sample_chars:break
        if chars<20000:raise RuntimeError('Not enough training text to fit a tokenizer')
        spm.SentencePieceTrainer.Train(input=str(sample),model_prefix=str(out/'spm'),
            vocab_size=vocab,model_type='bpe',character_coverage=0.9995,
            unk_id=0,eos_id=1,bos_id=-1,pad_id=-1,
            max_sentence_length=16384,hard_vocab_limit=False, num_threads=4)
        sample.unlink(missing_ok=True)
    sp=spm.SentencePieceProcessor(model_file=str(out/'spm.model'))
    if sp.vocab_size()>65535:raise ValueError('uint16 token array requires vocab<=65535')
    sizes={'train':int(max_train),'valid':int(valid_count),'test':int(test_count)}
    mm={}
    for k,n in sizes.items():
        mm[k]=np.memmap(out/f'{k}.bin',dtype=np.uint16,mode='w+',shape=(n,))
    pos={k:0 for k in sizes}
    doc_count={k:0 for k in sizes}
    print('Tokenizing attached FineWeb-Edu. Caps:',sizes,flush=True)
    for j,t in enumerate(documents(paths)):
        if stop_epoch is not None and time.time()>stop_epoch:
            print('Corpus preparation budget reached; using filled prefixes, if sufficient.',flush=True)
            break
        split=split_doc(t)
        remain=sizes[split]-pos[split]
        if remain<=0:
            if all(pos[k]>=sizes[k] for k in sizes):break
            continue
        ids=sp.encode(t,out_type=int)
        if not ids:continue
        ids.append(sp.eos_id())
        take=min(len(ids),remain)
        mm[split][pos[split]:pos[split]+take]=np.asarray(ids[:take],dtype=np.uint16)
        pos[split]+=take;doc_count[split]+=1
        if j and j%5000==0:print('data',j,'documents',pos,flush=True)
    for k in sizes:
        mm[k].flush();del mm[k]
        with (out/f'{k}.bin').open('r+b') as f:f.truncate(pos[k]*2)
    if pos['train']<512*128 or min(pos['valid'],pos['test'])<4096:
        raise RuntimeError('Not enough authentic FineWeb-Edu text before preparation deadline: '+str(pos))
    manifest={'complete':True,'root':str(out),'tokenizer':str(out/'spm.model'),
        'dataset':'HuggingFaceFW/fineweb-edu sample-10BT','data_provenance':source_manifest,
        'document_hash_split':'1% validation, 1% test, 98% train; exact hash by full text',
        'document_counts':doc_count,'sizes':pos,'vocab_size':sp.vocab_size(),
        'reused':False,'tokens_are_bpe':True,
        'limits':sizes,'sha256_tokenizer':hashlib.sha256((out/'spm.model').read_bytes()).hexdigest()}
    old.write_text(json.dumps(manifest,indent=2))
    return manifest


def load_tokens(manifest,part):
    f=Path(manifest['root'])/f'{part}.bin'
    return np.memmap(f,mode='r',dtype=np.uint16)

In [ ]:
from pathlib import Path

def available_prepared_dirs():
    bases=[]
    if PREPARED_DIR: bases.append(Path(PREPARED_DIR))
    for m in glob.glob('/kaggle/input/**/train.bin',recursive=True): bases.append(Path(m).parent)
    # Also reuse local data produced in an earlier run of the same notebook.
    bases.append(OUT/'data')
    return list(dict.fromkeys(bases))

def text_to_bytes_file(text):
    # Byte tokenizer avoids relying on an externally stored vocabulary for fallback.
    arr=np.frombuffer(text.encode('utf-8'),dtype=np.uint8).astype(np.uint16)
    t=int(len(arr)*0.90);v=int(len(arr)*0.95)
    if min(v-t,len(arr)-v)<8192: raise ValueError('Text file too short for validation and test')
    return {'train':arr[:t].copy(),'valid':arr[t:v].copy(),'test':arr[v:].copy()}, 256

def load_data():
    if DATA_MODE in ('auto','prepared'):
        for d in available_prepared_dirs():
            if all((d/x).exists() for x in ['train.bin','valid.bin','test.bin','spm.model']):
                import sentencepiece as spm
                sp=spm.SentencePieceProcessor(model_file=str(d/'spm.model'))
                arrays={split:np.memmap(d/(split+'.bin'),dtype=np.uint16,mode='r') for split in ('train','valid','test')}
                result={'mode':'prepared_fineweb','root':str(d),'vocab':sp.vocab_size(),'lengths':{k:len(v) for k,v in arrays.items()}}
                return arrays,sp.vocab_size(),result
        if DATA_MODE=='prepared':raise FileNotFoundError('Set PREPARED_DIR to directory containing .bin files + spm.model')
    if DATA_MODE in ('auto','fineweb_parquet'):
        paths=glob.glob(PARQUET_GLOB,recursive=True)
        if paths:
            # Reuse EXACT project's document hash split and SentencePiece preprocessing.
            if len(paths)>1:
                fw=[s for s in paths if 'fineweb' in s.lower() or 'cartman' in s.lower()]
                if len(fw)==1: paths=fw
                else:raise RuntimeError('More than one parquet found. Narrow PARQUET_GLOB to one FineWeb shard.')
            manifest=prepare(str(OUT),paths[0],vocab=4096,
                             max_train=MAX_PREP_TRAIN_TOKENS,valid_count=150000,test_count=150000)
            arrays={split:load_tokens(manifest,split) for split in ('train','valid','test')}
            return arrays,int(manifest['vocab_size']),{'mode':'fineweb_parquet','manifest':manifest,'lengths':{k:len(v) for k,v in arrays.items()}}
        if DATA_MODE=='fineweb_parquet':raise FileNotFoundError('Attach a FineWeb-Edu parquet via Add Input')
    path=Path(TEXT_FILE) if TEXT_FILE else None
    if path and path.exists():
        raw=path.read_text(encoding='utf-8')
        arrays,vocab=text_to_bytes_file(raw)
        return arrays,vocab,{'mode':'attached_text_byte_tokenizer','path':str(path),'lengths':{k:len(v) for k,v in arrays.items()}}
    if DATA_MODE not in ('auto','tiny_shakespeare'):
        raise RuntimeError('Unknown DATA_MODE: '+DATA_MODE)
    print('WARNING: No prepared FineWeb input. Using much smaller Tiny Shakespeare text, repeatedly sampled.')
    cache=OUT/'tiny_shakespeare.txt'
    if not cache.exists():
        urllib.request.urlretrieve('https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',cache)
    arrays,vocab=text_to_bytes_file(cache.read_text(encoding='utf-8'))
    return arrays,vocab,{'mode':'tiny_shakespeare_byte_tokenizer','path':str(cache),'lengths':{k:len(v) for k,v in arrays.items()},
                        'warning':'10M tokens are presentations with heavy reuse, NOT 10M distinct tokens'}

TOKENS,VOCAB,DATA_INFO=load_data()
assert min(map(len,TOKENS.values()))>=CONTEXT+8192
assert max(int(np.max(a)) for a in TOKENS.values())<VOCAB
DATA_INFO['train_sha256_first_1mb']=hashlib.sha256(np.asarray(TOKENS['train'][:524288],dtype=np.uint16).tobytes()).hexdigest()
safe_json_write(OUT/'data_info.json',DATA_INFO)
print('Dataset:',DATA_INFO)
print('Model vocabulary:',VOCAB)

## 8 · 10-million-token language training, checkpoints and validation

In [ ]:
def make_lm_batch(step,valid_tokens=CONTEXT*BATCH):
    train=TOKENS['train']
    max_window=(len(train)-CONTEXT-1)//CONTEXT
    rng=np.random.default_rng(SEED+1_000_003*int(step))
    picks=rng.integers(0,max_window,size=BATCH)
    w=np.stack([np.asarray(train[int(i)*CONTEXT:int(i)*CONTEXT+CONTEXT+1],dtype=np.int64) for i in picks])
    x=torch.from_numpy(w[:,:-1].copy()).to(DEVICE)
    y=torch.from_numpy(w[:,1:].copy()).to(DEVICE)
    if valid_tokens<CONTEXT*BATCH:
        y.reshape(-1)[valid_tokens:]=-100  # exact scored-token budget on final batch
    return x,y

@torch.no_grad()
def eval_lm(model,split='valid',max_tokens=SMALL_VAL_TOKENS):
    model.eval()
    arr=TOKENS[split]; windows=min(max_tokens//CONTEXT,(len(arr)-1)//CONTEXT)
    if windows<1:raise RuntimeError('Not enough validation text')
    nll=0.; right=0; seen=0
    for start in range(0,windows,BATCH):
        bs=min(BATCH,windows-start)
        w=np.stack([np.asarray(arr[(start+i)*CONTEXT:(start+i)*CONTEXT+CONTEXT+1],dtype=np.int64) for i in range(bs)])
        x=torch.from_numpy(w[:,:-1].copy()).to(DEVICE)
        y=torch.from_numpy(w[:,1:].copy()).to(DEVICE)
        logits=model(x)
        nll+=float(F.cross_entropy(logits.reshape(-1,VOCAB),y.reshape(-1),reduction='sum').item())
        right+=int((logits.argmax(-1)==y).sum().item())
        seen+=int(y.numel())
    model.train()
    loss=nll/seen
    return {'tokens':seen,'nll':loss,'perplexity':math.exp(loss),'top1_accuracy_pct':100*right/seen}


def lr_schedule(tokens_seen):
    warm=1_000_000; peak=8e-4; floor=8e-5
    if tokens_seen<warm:return peak*max(tokens_seen+CONTEXT,1)/warm
    frac=min(1.,max(0.,(tokens_seen-warm)/max(TARGET_TOKENS-warm,1)))
    return floor+(peak-floor)*0.5*(1+math.cos(math.pi*frac))

train_path=OUT/'checkpoint_latest.pt'
train_rows=[]
seen=0;step=0;elapsed_train=0.
model=PPLSTMLanguageModel(VOCAB).to(DEVICE)
opt=torch.optim.AdamW(model.parameters(),lr=8e-4,betas=(0.9,0.95),weight_decay=0.01)
resume=Path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else (train_path if train_path.exists() else None)
if resume and resume.exists():
    saved=torch.load(resume,map_location=DEVICE,weights_only=False)
    assert saved['vocab']==VOCAB and saved['context']==CONTEXT and saved['batch']==BATCH
    assert saved['data_fingerprint']==DATA_INFO['train_sha256_first_1mb'],'Dataset differs from checkpoint'
    model.load_state_dict(saved['model']);opt.load_state_dict(saved['optimizer'])
    seen=int(saved['seen_tokens']);step=int(saved['steps']);elapsed_train=float(saved['training_seconds'])
    train_rows=list(saved.get('training_history',[]))
    print('RESUMED from',resume,'after',seen,'scored tokens')
else:
    print('Fresh initialization. Parameters:',params(model))

# Main training graph captures at exactly this fixed shape.
example_x,example_y=make_lm_batch(step)
engine='graph' if LM_GRAPH_OK else 'eager'
try:
    graph_trainer=CapturedForwardBackward(model,example_x,example_y) if engine=='graph' else None
except Exception as ex:
    if TRAIN_GRAPH_REQUIRED:raise RuntimeError('Real LM graph capture failed; no hidden eager fallback') from ex
    engine='eager';graph_trainer=None
print('ACTUAL training engine:',engine,'(graph means captured forward+backward; AdamW runs outside capture)')

if STARTUP_SMOKE_ONLY:
    # Smoke only, no model update; test graph replay and validation.
    model.train()
    v=graph_trainer.replay(example_x,example_y) if graph_trainer else task_loss(model(example_x),example_y)
    torch.cuda.synchronize()
    print('SMOKE TRAIN LOSS',float(v.item()),'VAL',eval_lm(model,'valid',2048))
else:
    if TRAIN_LM and seen<TARGET_TOKENS:
        next_val=((seen//EVAL_EVERY_TOKENS)+1)*EVAL_EVERY_TOKENS
        window_start=time.perf_counter()
        while seen<TARGET_TOKENS:
            count=min(BATCH*CONTEXT,TARGET_TOKENS-seen)
            x,y=make_lm_batch(step,valid_tokens=count)
            lr=lr_schedule(seen)
            if engine=='graph': loss=update_graph_step(graph_trainer,opt,x,y,lr)
            else:loss=update_eager_step(model,opt,x,y,lr)
            step+=1; seen+=count
            if seen>=next_val or seen==TARGET_TOKENS:
                torch.cuda.synchronize()
                elapsed_train+=time.perf_counter()-window_start
                last_train_loss=float(loss.item())
                validation=eval_lm(model,'valid',SMALL_VAL_TOKENS)
                row={'scored_tokens':seen,'optimizer_steps':step,'train_loss':last_train_loss,
                     'validation_nll':validation['nll'],'validation_ppl':validation['perplexity'],
                     'validation_top1_pct':validation['top1_accuracy_pct'],
                     'validation_tokens':validation['tokens'],
                     'training_seconds_excluding_validation':elapsed_train,
                     'train_tokens_per_second':seen/max(elapsed_train,1e-8),
                     'learning_rate':lr,'engine':engine}
                train_rows.append(row)
                save_csv(OUT/'training_checkpoints.csv',train_rows)
                export_metric_bundle()
                print('TOKENS',f'{seen:,}','val_ppl',round(validation['perplexity'],2),'acc%',round(validation['top1_accuracy_pct'],2),
                      'tok/s',round(seen/max(elapsed_train,1e-8)),flush=True)
                # Save periodically and at the end; checkpoint is atomic.
                if seen//SAVE_EVERY_TOKENS != (seen-count)//SAVE_EVERY_TOKENS or seen==TARGET_TOKENS:
                    atomic_torch_save({'model':model.state_dict(),'optimizer':opt.state_dict(),'steps':step,
                       'seen_tokens':seen,'training_seconds':elapsed_train,'training_history':train_rows,
                       'vocab':VOCAB,'context':CONTEXT,'batch':BATCH,
                       'data_fingerprint':DATA_INFO['train_sha256_first_1mb'],
                       'engine':engine},train_path)
                    print('Saved resumable checkpoint:',train_path)
                next_val+=EVAL_EVERY_TOKENS
                window_start=time.perf_counter()

model.eval()
if STARTUP_SMOKE_ONLY:
    final_scores={'status':'smoke_only','training_tokens':0}
elif TRAIN_LM:
    if seen==TARGET_TOKENS:
        final_scores={'status':'completed','training_tokens':seen,'optimizer_steps':step,'parameters':params(model),
            'training_seconds_excluding_eval':elapsed_train,'training_tokens_per_second':seen/max(elapsed_train,1e-8),
            'engine':engine,'validation':eval_lm(model,'valid',FINAL_VAL_TOKENS),
            'test':eval_lm(model,'test',FINAL_VAL_TOKENS)}
        atomic_torch_save({'model':model.state_dict(),'vocab':VOCAB,'context':CONTEXT,
                           'source':'mLSTM+PPLSTM 4-block'},OUT/'model_final.pt')
    else:
        final_scores={'status':'partial','training_tokens':seen,'optimizer_steps':step,'engine':engine}
else:
    final_scores={'status':'training_disabled','training_tokens':seen,'engine':engine}
safe_json_write(OUT/'language_training_summary.json',final_scores)
print('LANGUAGE TRAINING SUMMARY:',final_scores)

## 9 · Summary and one-click results bundle

In [ ]:
CONFIG={k:globals()[k] for k in ['TARGET_TOKENS','WIDTH','DEPTH','HEADS','CONTEXT','BATCH','SEED',
    'TRAIN_GRAPH_REQUIRED','TRAIN_LM','RUN_PROBES','PROBE_STEPS','PROBE_TRAIN_CONTEXT',
    'PROBE_BATCH','PROBE_EVAL_CONTEXTS','PROBE_REPORT_EVERY','PROBE_FINAL_EVAL_BATCHES',
    'PROBE_SAVE_EVERY','DATA_MODE','STARTUP_SMOKE_ONLY']}
CONFIG['gpu']=torch.cuda.get_device_name(0)
CONFIG['torch']=torch.__version__
CONFIG['model']='4 xLSTM blocks: mLSTM PPLSTM mLSTM PPLSTM'
CONFIG['limitations']=['single seed','synthetic probes train independent models, NOT zero-shot LM tests',
    'CUDA Graph captures forward+backward only; clipping and AdamW remain eager',
    'no custom fused CUDA/Triton kernels or optimized sLSTM backend',
    'token presentations may reuse source text; check data_info.json']
safe_json_write(OUT/'run_config.json',CONFIG)
summary={'config':CONFIG,'data':DATA_INFO,'language':final_scores,
         'diagnostic_passes':sum(x['passed'] for x in DIAGNOSTICS),
         'diagnostic_tests':len(DIAGNOSTICS),'memory_probes':PROBE_ROWS}
safe_json_write(OUT/'summary.json',summary)
if PROBE_ROWS:
    display(pd.DataFrame(PROBE_ROWS)[['task','eval_context','accuracy_pct','chance_pct','engine','status']])
if train_rows:
    display(pd.DataFrame(train_rows).tail(5))

# Keep full model checkpoints separate from the compact CSV/JSON metrics archive.
zip_path=export_metric_bundle()
print('RESULTS ARCHIVE:',zip_path)
print('RESUMABLE TRAINING CHECKPOINT:',train_path,'(save it separately if you plan to resume)')
print('Final model:',OUT/'model_final.pt')
print('If Kaggle stops: Save Version, then attach its output as Input and set RESUME_CHECKPOINT.')

### How to download

After **Save Version / Save & Run All**, open the notebook's **Output** files and download `pplstm_10m_memory_5000step/PPLSTM_Kaggle_10M_5000Step_Memory_Results.zip`. Upload it here for analysis. The model and optimizer checkpoint files are in the same output directory but intentionally not included in the small metrics ZIP.

**To resume interrupted training:** Save Version or download the output checkpoints, attach the saved outputs as a Kaggle input, and restore the appropriate `checkpoint_latest.pt` / `probe_<task>_latest.pt` under the notebook output directory before running the training cells. Each probe's 5,000 steps are separate from the language model's 10M scored token budget.

**Important:** a synthetic task's evaluation at 256 or 512 tokens is an **out-of-training-length generalization** measurement, not a guarantee that PPLSTM carries a hidden state indefinitely. This implementation evaluates complete contexts and does not supply a streaming KV-cache/state interface.